# Marco de Aprendizaje por Refuerzo Profundo para la Gestión de Portafolios Financieros

En los mercados financieros, la gestión automatizada de portafolios es un desafío crítico que combina diversas áreas como las matemáticas y economía. Tradicionalmente, este problema se ha abordado con estrategias basadas en reglas o modelos de optimización estática. Sin embargo, el aprendizaje por refuerzo profundo (*Deep Reinforcement Learning*, DRL) emerge como un enfoque prometedor al permitir que un agente aprenda políticas de inversión óptimas directamente de los datos, adaptándose dinámicamente a las condiciones del mercado.

Este proyecto implementa y extiende el marco propuesto por **Jiang et al. (2017)** en el artículo _["A Deep Reinforcement Learning Framework for the Financial Portfolio Management Problem"](https://arxiv.org/abs/1706.10059)_. El objetivo principal es desarrollar un agente autónomo capaz de asignar capital entre múltiples activos (acciones del S&P 500 o criptomonedas) para maximizar el rendimiento acumulado, considerando costos de transacción y riesgos.

## Componentes Clave del Proyecto

### 1. Preprocesamiento de Datos
- Los datos brutos (precios *Open, High, Low, Close*) se transforman en matrices normalizadas que capturan relaciones temporales clave, como:
  - Proporciones entre precios (*Close(t-1)/Open(t-1)*, *High(t-1)/Open(t-1)*, etc.).
- Se adaptan dos conjuntos de datos:
  - **S&P 500**: Datos diarios con 4 características por acción.
  - **Criptomonedas (Poloniex)**: Datos de mercados 24/7 con 3 características (dado que *Close(t) = Open(t+1)*).

### 2. Entorno de Trading (RL Environment)
- El entorno simula un mercado real con:
  - **Estados**: Historial de precios (ventana temporal de 50 pasos) y distribución actual del portafolio.
  - **Acciones**: Vector de pesos de inversión (asignación de capital entre activos).
  - **Recompensas**: Rendimiento relativo ajustado por costos de transacción y un *baseline* (estrategia equi-ponderada).
- Incluye mecanismos como tasas de interés para efectivo no invertido y penalizaciones por concentración de riesgo.

### 3. Arquitectura del Agente (Deep Policy Network)
- Utiliza una red neuronal convolucional para procesar los tensores de datos y extraer patrones temporales.
- La salida es una distribución de probabilidad (softmax) sobre los pesos del portafolio, optimizada mediante políticas de gradiente ascendente (*Policy Gradient*).

### 4. Desafíos y Mejoras Propuestas
- **Sensibilidad a Hiperparámetros**: El agente tiende a ser conservador en cambios de posición, lo que sugiere ajustar costos de transacción o exploración.
- **Alternativas**: Uso de espacios de acción discretos o funciones de recompensa más sofisticadas para incentivar dinamismo.

## Contribución y Relevancia
Este proyecto no solo replica el trabajo original, sino que lo extiende a nuevos contextos (mercados de acciones tradicionales) y explora limitaciones prácticas. Su marco es escalable a otros activos y frecuencias de trading, ofreciendo un laboratorio para robar estrategias basadas en *DRL* en finanzas.


# Preprocesamiento de los datos

##  Archivo data_pipe.ipynb
Aquí se transforman datos históricos de acciones del S&P 500 en matrices normalizadas para entrenamiento de RL.

### **Pasos**
1. **Carga de datos**:
   - Lee archivos CSV individuales para cada acción (505 stocks disponibles)
   - Filtra acciones con datos incompletos
   - Mantiene solo acciones con 1259 registros (5 años de datos diarios)

   Lo hace en esta parte:
   ```python
   for s in tqdm(stocks):
    df = pd.read_csv(os.getcwd() + data_dir + s)
        
    if len(df)!=1259:
        
        not_kept_stocks.append(s)
    else:
        kept_stocks.append(s)
    ```

2. **Transformación de características**:
   - Calcula 4 ratios financieros por acción:
     - `Close(t-1)/Open(t-1)`  
     - `High(t-1)/Open(t-1)`  
     - `Low(t-1)/Open(t-1)`  
     - `Open(t)/Open(t-1)`
   - Normaliza implícitamente

   Lo hace aquí:
   ```python
   array_open = np.transpose(np.array(list_open))[:-1]           # Open(t-1)
   array_open_next = np.transpose(np.array(list_open))[1:]       # Open(t)

    X = np.transpose(np.array([
      array_close/array_open,     # Close(t-1)/Open(t-1)
      array_high/array_open,      # High(t-1)/Open(t-1)
      array_low/array_open,       # Low(t-1)/Open(t-1)
      array_open_next/array_open  # Open(t)/Open(t-1)
    ]), axes=(0, 2, 1))  # Reorganiza a (features, stocks, timesteps)

    ```

3. **Estructura de salida**:
   - Matriz con shape `print(X.shape)`
   - Ejemplo: Para 5 acciones seleccionadas → `(4, 5, 1258)`

**Uso en RL**:  
Proporciona los estados (matrices) para el agente, representando la evolución histórica de los precios.

---

## Archivo data_pipe_poloniex.ipynb

Adaptar el pipeline a datos de criptomonedas (mercado 24/7) manteniendo compatibilidad con el entorno de RL.

### **Pasos**:

1. **Carga de datos del mercado 24/7**:
   - Utilizarán Close(t) = Open(t+1) → Elimina la columna `Close` porque el mercado nunca cierra
   - Se quedarán con solo 3 features:
     - `High(t-1)/Open(t-1)`  
     - `Low(t-1)/Open(t-1)`  
     - `Open(t)/Open(t-1)`

  ```python
  for s in stocks:
      df = pd.read_csv('.'+data_dir+s)
      print(s, len(df))
  ```

2. **Filtrado temporal**:
   - Descarta criptomonedas con menos de 17,000 puntos (≈1 año de datos)
   - Alinea series temporales usando el mínimo común (`min_len = 17,031`)

   ```python
    kept_stocks = ['ETCBTC.csv', 'ETHBTC.csv', 'DOGEBTC.csv', 'ETHUSDT.csv', 'BTCUSDT.csv',
              'XRPBTC.csv', 'DASHBTC.csv', 'XMRBTC.csv', 'LTCBTC.csv', 'ETCETH.csv']
    len_stocks = list()
    len_stocks = [len(pd.read_csv('.'+data_dir+s)) for s in kept_stocks]
    min_len = np.min(len_stocks)  # Ej: 17,031 puntos (≈1 año)
    ```

3. **Estructura de salida**:
   - Se quedán con sólo las 3 features y quitan Close
    ```python
      X = np.transpose(np.array([
        array_high/array_open,      # High(t-1)/Open(t-1)
        array_low/array_open,       # Low(t-1)/Open(t-1)
        array_open_next/array_open  # Open(t)/Open(t-1)
        ]), axes=(0, 2, 1))
    ```
   - Se quedan con una matriz con shape `(features, criptomonedas, timesteps)`



# Entorno de Trading para RL (enviroment.py)

Se implementa un entorno de Gym (libreria de python par RL) personalizado para simular un mercado financiero donde un agente de RL gestiona un portafolio de activos. Maneja:
- Estados: Datos históricos de precios + distribución actual del capital.
- Acciones: Rebalanceo de pesos de inversión.
- Recompensas: Rentabilidad ajustada por costos y riesgo.

## Estructura importante del código

1. **Inicialización (`__init__`)**  

Configura parámetros críticos y carga datos preprocesados:
```python
class TradeEnv:
    def __init__(self, path='./np_data/input.npy', window_length=50,
                 portfolio_value=10000, trading_cost=0.25/100,
                 interest_rate=0.02/250, train_size=0.7):
        
        self.data = np.load(path)  # Matrices que procesamos con los datos
        self.window_length = window_length  # Ventana histórica
        self.trading_cost = trading_cost    # Costo por transacción (0.25%)
        self.interest_rate = interest_rate  # Tasa para efectivo no invertido
```

2. ** Método `step`**

Ejecuta una acción (nuevos pesos) y calcula el nuevo estado/recompensa:

```python
def step(self, action):
    # 1. Extrae pesos anteriores y valor del portafolio
    w_previous = self.state[1]  # Pesos
    pf_previous = self.state[2]  # Valor total

    # 2. Calcula costos de transacción ||w_new - w_old||
    cost = pf_previous * np.linalg.norm(action - w_previous, ord=1) * self.trading_cost
    
    # 3. Aplica evolución de precios (returns diarios)
    update_vector = self.readUpdate(self.index)  # Vector de returns (ej: [1.02, 0.98, 1.05])
    v_evol = (pf_previous - cost) * action * update_vector
    
    # 4. Nuevo valor del portafolio y pesos
    pf_evol = np.sum(v_evol)
    w_evol = v_evol / pf_evol  # Normaliza a pesos
    
    # 5. Calcula recompensa (rendimiento porcentual)
    reward = (pf_evol - pf_previous) / pf_previous
    
    return new_state, reward, done
```

3. ** Método `reset`**

Reinicia el entorno a un estado inicial:

```python
def reset(self, w_init, p_init, t=0):
    # Estado inicial: (datos históricos, pesos iniciales, valor inicial)
    self.state = (self.readTensor(self.data, self.window_length), w_init, p_init)
    return self.state
```

4. **Manejo de estados**

El estado contiene `(Matriz, pesos, valor)`

```python
# Ejemplo de estado para 3 activos y ventana de 50 días:
state = (
    np.array(shape=(3, 3, 50)),  # 3 features, 3 activos, 50 días
    np.array([0.1, 0.1, 0.8]),    # Pesos: 10% A, 10% B, 80% C
    10000.0                       # Valor total: $10,000
)
```

# Entrenamiento y evalución para RL (DPM.ipynb)

1. **Configuración Inicial**
  - **Hiperparámetros y Datos**:

  ```python
  # Parámetros del dataset y entorno
  path_data = './np_data/inputCrypto.npy'  # Ruta a datos preprocesados
  data = np.load(path_data) #lectura de datos
  trading_period = data.shape[2]
  nb_feature_map = data.shape[0]
  nb_stocks = data.shape[1]

  # Arquitectura de la red
  dict_hp_net = {'n_filter_1': 2, 'n_filter_2': 20, 'kernel1_size': (1, 3)}
  dict_hp_pb = {'batch_size': 50, 'ratio_train': 0.6,'ratio_val': 0.2, 'length_tensor': 10,
              'ratio_greedy':0.8, 'ratio_regul': 0.1}
  .
  .
  .
  ```

2. **Creación de entornos**

Se instancian múltiples entornos para comparar estrategias:

```python
# Entorno principal para el agente RL
env = TradeEnv(path=path_data, window_length=n,
               portfolio_value=pf_init_train, trading_cost=trading_cost,
               interest_rate=interest_rate, train_size=dict_hp_pb['ratio_train'])

# Baseline: Portafolio equi-ponderado
env_eq = TradeEnv(...)  # Mismos parámetros, acción fija w_eq = [1/(m+1), ...]

# Estrategia conservadora (solo efectivo)
env_s = TradeEnv(...)   # Acción fija w_s = [1, 0, ..., 0]

# Entornos para estrategias "full stock" (inversión en una sola acción)
for i in range(m):
    action = np.array([0]*(i+1) + [1] + [0]*(m-(i+1)))
    env_fu.append(TradeEnv(...))  # Ej: 100% en ETHBTC
```

3. **Arquitectura de Policy Network**

Clase `Policy` que define la red neuronal convolucional:
  
```python
class Policy(object):
    def __init__(self, ...):
        # Capas convolucionales para procesar tensores de precios
        self.conv1 = tf.layers.conv2d(inputs=..., filters=2, kernel_size=(1,3))
        self.conv2 = tf.layers.conv2d(inputs=..., filters=20, strides=(50,1))
        
        # Softmax para generar pesos del portafolio
        self.action = tf.nn.softmax(...)
        
        # Cálculo de recompensa ajustada
        self.reward = (rendimiento_agente - rendimiento_baseline - penalización_riesgo)
```

4. **Memoria (PVM)**

Almacena historial de pesos para entrenamiento por batches:

```python
class PVM:
    def __init__(self, ...):
        self.memory = np.array([w_init]*total_steps)  # Inicialización
        
    def draw(self, beta=sample_bias):
        # Muestreo geométrico para diversidad de batches
      while 1:
        z = np.random.geometric(p=beta)
        tb = self.total_steps - self.batch_size + 1 - z
        if tb >= 0:
          return tb # Índice de inicio del batch
```
5. **Ciclo de entrenamiento**

```python
for e in range(n_episodes):
    memory = PVM(...)  # Reinicia memoria por episodio
    for nb in range(n_batches):
        i_start = memory.draw()  # Batch inicial
        state, _ = env.reset(memory.get_W(i_start), ...)
        
        # Interacción con el entorno
        for bs in range(batch_size):
            action = actor.compute_W(X_t, W_previous)  # Política actual
            state, reward, done = env.step(action)
            
            # Actualiza memoria y entrena
            memory.update(i_start + bs, new_weights)
            actor.train(batch_X, batch_W, batch_rewards)
```

6. **Evaluación y rendimiento**

  - **Rendimiento vs Baselines**

  ```python
  plt.plot(p_list, label='Agente RL')
  plt.plot(p_list_eq, label='Equi-ponderado')
  plt.plot(p_list_s, label='Efectivo')
  plt.plot(p_list_fu[i], label='Full Stock X')  # Para cada acción
  ```
  - **Evolución de pesos**

  ```python
  plt.bar(np.arange(m+1), weights, labels=['Efectivo'] + list_stock)
  ```


# Conclusión

## **Logros Clave**  
1. **Implementación Exitosa de un Marco de RL**:  
   - Se replicó y extendió el marco propuesto por Jiang et al. (2017), demostrando cómo el **aprendizaje por refuerzo profundo (DRL)** puede aplicarse a la gestión de portafolios en mercados tradicionales (S&P 500) y criptomonedas (Poloniex).  
   - El agente aprendió a asignar pesos de inversión dinámicamente, considerando costos de transacción, tasas de interés, y volatilidad del mercado.

2. **Adaptabilidad a Diferentes Activos**:  
   - Los pipelines de preprocesamiento (`data_pipe.ipynb` y `data_pipe_poloniex.ipynb`) permitieron manejar datos con características distintas (ej: mercados 24/7 en criptomonedas vs. horarios fijos en acciones).  
   - El entorno (`environment.py`) fue flexible para integrar ambos tipos de datos, manteniendo consistencia en los estados y recompensas.

3. **Arquitectura Innovadora**:  
   - La red neuronal convolucional procesó matrices históricos de precios (ventanas de 50 timesteps) para extraer patrones temporales, superando enfoques estáticos.  
   - La función de recompensa ajustada (`reward = rendimiento_agente - baseline - penalización_riesgo`) incentivó estrategias balanceadas entre rentabilidad y diversificación.

## **Desafíos y Limitaciones**  
1. **Sensibilidad a Hiperparámetros**:  
   - El rendimiento del agente dependió críticamente de valores como `trading_cost`, `ratio_greedy`, y el tamaño de la ventana histórica. Pequeños cambios podrían generar políticas demasiado conservadoras o arriesgadas.

2. **Entrenamiento y Estabilidad**:  
   - El proyecto aún no alcanza resultados consistentemente superiores a los baselines (ej: equi-ponderado), posiblemente por la complejidad de optimizar políticas en mercados estocásticos.  
   - La exploración vs. explotación (manejada con `ratio_greedy`) requirió ajustes manuales, sugiriendo la necesidad de métodos como **ε-greedy adaptativo** o **entropía en la política**.

3. **Escalabilidad**:  
   - El enfoque convolucional funcionó bien para 5-10 activos, pero podría enfrentar desafíos computacionales en portafolios más grandes (ej: 100+ acciones).

##  **Conclusión Final**  
Este proyecto demuestra que el **aprendizaje por refuerzo profundo** es una herramienta prometedora para la gestión automatizada de portafolios, capaz de adaptarse a condiciones de mercado dinámicas y aprender políticas de inversión sin depender de supuestos simplificadores. Si bien enfrenta desafíos técnicos (estabilidad del entrenamiento, hiperparámetros), su capacidad para integrar datos heterogéneos y optimizar decisiones secuenciales lo posiciona como un enfoque disruptivo en finanzas cuantitativas.  
